## scRNA preprocessing

### Import libraries and setup files

In [ ]:
import sys, os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

gene_name_mapping = "references/t2g.txt"


In [ ]:
from typing import List, Dict
import anndata as ad
from anndata import AnnData

def build_adata(h5ads_map: Dict) -> AnnData:
    """
    Builds an AnnData object from a previously downloaded and extracted IGVF tar.gz file.
    Assumes files live under data/{my_file}/ and uses global paths for metadata CSVs.
    """

    adatas = {}

    for sample_id, filename in h5ads_map.items():
        try:
            sample_adata = sc.read(filename)
        except FileNotFoundError:
            print(f"File not found: {filename}")
            continue
        sample_adata.var_names_make_unique()
        adatas[sample_id] = sample_adata

    adata = ad.concat(adatas, label="sample")
    adata.obs_names_make_unique()

    return adata

def add_gene_names_to_adata(adata: AnnData, gene_metadata: str) -> AnnData:
    """
    Add gene names to an AnnData object from a gene metadata TSV.
    """
    gene_df = pd.read_csv(gene_metadata, header=None, sep="\t", names=['transcript_id', 'gene_id', 'gene_name', 'gene_name_unique', 'chromosome', 'start', 'end', 'strand'])
    # Keep only gene_id and gene_name and remove duplicates
    gene_df = gene_df[['gene_id', 'gene_name']].drop_duplicates()
    gene_df = gene_df.set_index('gene_id')
    adata.var['gene_name'] = adata.var.index.map(gene_df['gene_name'])
    return adata

### Create scanpy object and calculate basic QCs.

This assumes you have run the download RNA h5ads script. Here we create a scanpy object with no filtering and plot the RNA knee plot. 

Then we apply a very minimal filter of 10 UMIs per barcode and calculate some basic QCs per barcode to plot. 

In [ ]:
h5ad_map = {
    "Subpool-1": "data/IGVFFI0500LLBC.h5ad",
    "Subpool-2": "data/IGVFFI8092YGSW.h5ad",
    "Subpool-3": "data/IGVFFI0400PXGY.h5ad",
    "Subpool-4": "data/IGVFFI5643DMQY.h5ad",
    "Subpool-5": "data/IGVFFI9719OAKV.h5ad"

}

adata = build_adata(h5ad_map)
adata.obs["AnalysisSet"] = adata.obs["sample"]
adata.obs["cell_barcode"] = adata.obs.index

print(adata.obs["sample"].value_counts())

add_gene_names_to_adata(adata, gene_metadata=gene_name_mapping)
adata.obs.index = adata.obs['cell_barcode']
adata.var['gene_id'] = adata.var.index
adata.var_names = adata.var['gene_name']
adata.var_names_make_unique()

#change gene_name column title with gene_name_unique
adata.var.rename(columns={'gene_name': 'gene_name_unique'}, inplace=True)


In [ ]:
# QC scoring

#Keep barcodes with UMIs >=10
adata = adata[adata.X.sum(axis=1).A1 >= 100]

# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("mt-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo"], inplace=True, log1p=True
)

print(adata.obs["sample"].value_counts())

#adata = adata[adata.obs['total_counts'] < max_umi, :]
print("<150000 UMIs:", adata.shape, flush=True)

#adata = adata[adata.obs['n_genes_by_counts'] > min_genes, :] # min number of genes per cell
print(">250 genes:", adata.shape, flush=True)

#adata = adata[adata.obs['pct_counts_mt'] < max_mito, :]
print("<1% mt:", adata.shape, flush=True)

gc.collect()

In [ ]:
#This will be used in the joint plot scatterplot to calculate overlap with ATAC.
adata.obs.to_csv("results/filt_rna_100UMI.barcodes_metrics.tsv", sep = "\t")


In [ ]:
adata.write_h5ad("results/filt_rna_100UMI.h5ad")

In [ ]:
#violin plots

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts"],
    groupby='sample',
    size=0,
    rotation = 90,
    multi_panel = True,
    show = True,
    log = True
)

In [ ]:
#violin plots

sc.pl.violin(
    adata,
    ["pct_counts_mt","pct_counts_ribo"],
    groupby='sample',
    size=0,
    rotation = 90,
    multi_panel = True,
    show = True,
    log = False
)